In [0]:
from pyspark.sql.functions import col, to_date, row_number, expr
from pyspark.sql.window import Window

data = [
    (1, '1/20/2025'),
    (1, '1/21/2025'),
    (1, '1/22/2025'),
    (1, '1/24/2025'),
    (2, '1/15/2025'),
    (2, '1/16/2025'),
    (2, '1/18/2025'),
    (3, '1/15/2025'),
    (3, '1/16/2025'),
    (3, '1/17/2025'),
    (3, '1/18/2025')
]

columns = ["emp_id", "login_date"]

df = spark.createDataFrame(data, columns)

In [0]:
df.display()

emp_id,login_date
1,1/20/2025
1,1/21/2025
1,1/22/2025
1,1/24/2025
2,1/15/2025
2,1/16/2025
2,1/18/2025
3,1/15/2025
3,1/16/2025
3,1/17/2025


In [0]:
win_spec = Window.partitionBy("emp_id").orderBy("login_date")
df1 = df.withColumn("row_num", row_number().over(win_spec))
df1.display()

emp_id,login_date,row_num
1,1/20/2025,1
1,1/21/2025,2
1,1/22/2025,3
1,1/24/2025,4
2,1/15/2025,1
2,1/16/2025,2
2,1/18/2025,3
3,1/15/2025,1
3,1/16/2025,2
3,1/17/2025,3


In [0]:
df2 = df1.withColumn("login_date", to_date(col("login_date"),"M/dd/yyyy"))
df2.display()

emp_id,login_date,row_num
1,2025-01-20,1
1,2025-01-21,2
1,2025-01-22,3
1,2025-01-24,4
2,2025-01-15,1
2,2025-01-16,2
2,2025-01-18,3
3,2025-01-15,1
3,2025-01-16,2
3,2025-01-17,3


In [0]:
df3 = df2.withColumn("new_date", col("login_date")-col("row_num"))
df3.display()

emp_id,login_date,row_num,new_date
1,2025-01-20,1,2025-01-19
1,2025-01-21,2,2025-01-19
1,2025-01-22,3,2025-01-19
1,2025-01-24,4,2025-01-20
2,2025-01-15,1,2025-01-14
2,2025-01-16,2,2025-01-14
2,2025-01-18,3,2025-01-15
3,2025-01-15,1,2025-01-14
3,2025-01-16,2,2025-01-14
3,2025-01-17,3,2025-01-14


In [0]:
df4 = df3.groupBy("emp_id" , "new_date").count().filter(col("count") >=3).select("emp_id")
df4.display()

emp_id
1
3
